<h1> Regridding 2km SST files to 9km to match NASA data<h1>

<h3>Import packages<h1>

In [1]:
import os
import subprocess
from netCDF4 import Dataset
from datetime import timedelta, datetime
import sys
from dateutil.parser import parse
import warnings
warnings.filterwarnings('ignore')
import numpy.ma as ma

<h3>Create global variables<h3>

In [2]:
ROOT_DIR = ("/Users/madisonrichardson/netpp")
SST_2k_DIR_T = os.path.join(ROOT_DIR, "data/{}/sst_2k")
SST_DIR_T = os.path.join(ROOT_DIR, "data/{}/sst")
WORK_DIR = os.path.join(ROOT_DIR, 'work')
BIN_DIR = os.path.join(ROOT_DIR, 'bin')
RES_DIR = os.path.join(ROOT_DIR, 'resources')
CWUTL_DIR = '"/Applications/CoastWatch Utilities/bin"'
CWUTIL_OFILE_HDF = 'nasa_9k.hdf'
CWUTIL_OFILE_NC = 'nasa_9k.nc'
MASTER_FILE = 'master_file_sst.nc'
CDL_IN_FILE = 'temp21.cdl'
TEMP_OUT_FILE = 'temp21.nc'

<h3>Define functions for sourcing data<h3>

In [3]:
def generate_url(date_obj):
    """Create a URL for downloading a data file.

    Creates a URL for downloading a data file from
    NOAA ACSPO Daily Global 0.02 degree ridded Super-collated SST
    data on a specific date.

    Args_:
        date_obj (datetime): A datetime object representing the specified date
                            for which the SST (sea surface temperature) data
                            URL is generated.

    Returns_:
        tuple: A tuple containing:
            - url (str): The complete URL to download the specified data file.
            - file_name (str): The name of the data file to be downloaded.
    """
    year = '{0:%Y}'.format(date_obj)
    doy = '{0:%j}'.format(date_obj)

    date_str = '{0:%Y%m%d}'.format(date_obj)

    base_url = (
        "https://coastwatch.noaa.gov/erddap/files/"
        "noaacwLEOACSPOSSTL3SKDaily/"
    )

    file_name = (
        f"{date_str}120000-STAR-L3S_GHRSST-SSTsubskin-"
        "LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc"
    )

    url = f"{base_url}{year}/{doy}/{file_name}"

    return url, file_name

In [4]:
def download_sst_file(url, SST_2k_DIR):
    """Download a file using generate_url_batch output.

    Downloads a file from the specified URL that is generated from
    generate_url_batch function and saves it to the specified
    directory.

    Args_:
        url (str): The URL of the file to be downloaded.
        SST_2k_DIR (str): The directory where the downloaded
        file will be saved.

    """
    command = f"wget -P {SST_2k_DIR} {url}"

    subprocess.call(command, shell=True)

    print(f"Downloaded: {url} to {SST_2k_DIR}")

<h3> Run main function to source data, regrid, and convert to Celsius<h3>

In [5]:
def main():
    """Run main function."""

    # Main code for downloading data and calculations for NetPP algorithm
    # Set the parameters for the code
    start_date = "2023-05-01"
    end_date = "2024-05-07"
    sensor = 'noaa21' # Either noaa20 or noaa21
    overwrite = True # Change to True if you want to overwrite existing NetCDF file

    # Parse the start and end dates
    start_date = parse(start_date)
    end_date = parse(end_date)

    # Determine range of years
    started_year = start_date.year
    ended_year = end_date.year

    sensors = ['noaa20', 'noaa21'] if sensor == 'both' else [sensor]

    for sensor in sensors:  # Loop over sensors
        current_date = start_date

        # Create directories for all years between start_year and end_year
        for year in range(started_year, ended_year + 1):
            # Create dynamic directories based on sensor and year range
            SST_2k_DIR = os.path.join(SST_2k_DIR_T.format(sensor))
            SST_DIR = os.path.join(SST_DIR_T.format(sensor))

            DIR_LIST = [ROOT_DIR,
                        WORK_DIR,
                        SST_2k_DIR,
                        SST_DIR,
                        ]

            for dr in DIR_LIST:
                os.makedirs(dr, exist_ok=True)
            print(len(DIR_LIST), 'directories validated')

        # Check for correct date values
        if start_date > end_date:
            sys.exit("start date must be < end date")

        while current_date <= end_date:
            print('Processing this date', current_date)

            # Define the output NetCDF file for the current date
            formatted_date = current_date.strftime('%Y%m%d')
            nc_filename = f'sst_leo_9km_{formatted_date}_daily.nc'

            odir = os.path.join(SST_DIR, str(current_date.year))
            os.makedirs(odir, exist_ok=True)

            nc_file_path = os.path.join(odir, nc_filename)

            # Add logic to not overwrite existing files unless argparse -o
            if os.path.isfile(nc_file_path):
                if not overwrite:
                    print(nc_filename, 'already exists')
                    current_date += timedelta(days=1)
                    continue
                else:
                    print('overwriting', nc_filename)

            date_noon = current_date.replace(hour=12, minute=0, second=0)
            timestamp = date_noon.timestamp()

            odir2 = os.path.join(SST_2k_DIR, str(current_date.year))
            os.makedirs(odir2, exist_ok=True)

            # Use a try to catch when data downloads fail
            try:
                # SST download and processing
                sst_url, sst_file = generate_url(current_date)
                download_sst_file(sst_url, odir2)
            except Exception as e:
                print('The NOAA file did not download', e)
                current_date += timedelta(days=1)
                continue

            # Regrid steps here
            formatted_date = current_date.strftime('%Y%m%d')
            ifile = os.path.join(
                odir2,
                f'{formatted_date}120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily'
                '-ACSPO_V2.81-v02.0-fv01.0.nc'
            )

            myCmd = ' '.join([os.path.join(CWUTL_DIR, 'cwregister2'),
                            '--clobber',
                            '--match=' + 'sea_surface_temperature',
                            '--master=' + os.path.join(RES_DIR, MASTER_FILE),
                            os.path.join(odir2, ifile),
                            os.path.join(WORK_DIR, CWUTIL_OFILE_HDF)
                            ])
            print(myCmd)
            print('Regrid NASA file',
                subprocess.call(myCmd, shell=True))

            cw_cmd = ' '.join([os.path.join(CWUTL_DIR, 'cwangles'),
                            '--location --float --units=deg',
                            os.path.join(WORK_DIR, CWUTIL_OFILE_HDF)
                            ])
            print('Add lat/lon', subprocess.call(cw_cmd, shell=True))

            # Convert hdf to netCDF with CW utilities

            myCmd = ' '.join([os.path.join(CWUTL_DIR, 'cwexport'), '-v',
                            os.path.join(WORK_DIR, CWUTIL_OFILE_HDF),
                            os.path.join(WORK_DIR, CWUTIL_OFILE_NC)
                            ])
            print('Convert  to NetCDF',
                subprocess.call(myCmd, shell=True))

            # Load datasets
            sst_ds = Dataset(os.path.join(WORK_DIR, CWUTIL_OFILE_NC), 'r')

            sst = sst_ds['sea_surface_temperature'][0, 0, :, :]
            sst = sst - 273.15

            sst_ds.close()

            # Generate output template file from cdl file
            myCmd = ' '.join(['ncgen',
                            '-o',
                            os.path.join(WORK_DIR, TEMP_OUT_FILE),
                            os.path.join(RES_DIR, CDL_IN_FILE)])
            print('Run ncgen',
                subprocess.call(myCmd, shell=True))

            # Open output template file in append mode
            nc_file = Dataset(os.path.join(WORK_DIR, TEMP_OUT_FILE),
                            'a',
                            format='NETCDF4')


            # Write sst, chlorophyll, par, and PPeu data to the netCDF file
            nc_file['sea_surface_temperature'][0, :, :] = sst[:, :]
            nc_file['time'][0] = timestamp

            # Modify metadata
            formatted_date_start = current_date.strftime('%Y-%m-%dT00:00:00Z')
            formatted_date_end = current_date.strftime('%Y-%m-%dT23:59:59Z')
            nc_file.time_coverage_start = formatted_date_start
            nc_file.time_coverage_end = formatted_date_end
            nc_file.title = ', '.join([
                "NOAA LEO SST"
                ])

            # Close netCDF file for this date
            nc_file.close()

            # Compress and archive
            myCmd = ' '.join(['nccopy',
                            '-d6',
                            os.path.join(WORK_DIR, TEMP_OUT_FILE),
                            os.path.join(WORK_DIR, nc_filename)
                            ])
            print('Compress ofile',
                subprocess.call(myCmd, shell=True))

            myCmd = ' '.join(['mv',
                            os.path.join(WORK_DIR, nc_filename),
                            nc_file_path
                            ])
            print('Archive ofile',
                subprocess.call(myCmd, shell=True))

            # Print where netCDF files were saved
            print(f"NetCDF file '{nc_filename}' archived at {nc_file_path}")

            # Add code to send to ERDDAP

            current_date += timedelta(days=1)


if __name__ == '__main__':
    main()


4 directories validated
4 directories validated
Processing this date 2023-05-01 00:00:00
overwriting sst_leo_9km_20230501_daily.nc


--2024-10-25 10:16:33--  https://coastwatch.noaa.gov/erddap/files/noaacwLEOACSPOSSTL3SKDaily/2023/121/20230501120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc
Resolving coastwatch.noaa.gov (coastwatch.noaa.gov)... 140.90.107.14, 140.90.107.15
Connecting to coastwatch.noaa.gov (coastwatch.noaa.gov)|140.90.107.14|:443... connected.
HTTP request sent, awaiting response... 200 
Length: 440897346 (420M) [application/x-netcdf]
Saving to: ‘/Users/madisonrichardson/netpp/data/noaa21/sst_2k/2023/20230501120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc.1’

     0K .......... .......... .......... .......... ..........  0%  207K 34m41s
    50K .......... .......... .......... .......... ..........  0% 2.25M 18m54s
   100K .......... .......... .......... .......... ..........  0% 4.19M 13m9s
   150K .......... .......... .......... .......... ..........  0%  713K 12m23s
   200K .......... .......... .......... .......... ..........  0% 48.7M 9m56s
   

Downloaded: https://coastwatch.noaa.gov/erddap/files/noaacwLEOACSPOSSTL3SKDaily/2023/121/20230501120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc to /Users/madisonrichardson/netpp/data/noaa21/sst_2k/2023
"/Applications/CoastWatch Utilities/bin"/cwregister2 --clobber --match=sea_surface_temperature --master=/Users/madisonrichardson/netpp/resources/master_file_sst.nc /Users/madisonrichardson/netpp/data/noaa21/sst_2k/2023/20230501120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc /Users/madisonrichardson/netpp/work/nasa_9k.hdf
Regrid NASA file 0
Add lat/lon 0


[INFO] Creating output /Users/madisonrichardson/netpp/work/nasa_9k.nc
[INFO] Writing sea_surface_temperature
[INFO] Writing latitude
[INFO] Writing longitude


Convert  to NetCDF 0
Run ncgen 0
Compress ofile 0
Archive ofile 0
NetCDF file 'sst_leo_9km_20230501_daily.nc' archived at /Users/madisonrichardson/netpp/data/noaa21/sst/2023/sst_leo_9km_20230501_daily.nc
Processing this date 2023-05-02 00:00:00
overwriting sst_leo_9km_20230502_daily.nc


...

Downloaded: https://coastwatch.noaa.gov/erddap/files/noaacwLEOACSPOSSTL3SKDaily/2023/122/20230502120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc to /Users/madisonrichardson/netpp/data/noaa21/sst_2k/2023
"/Applications/CoastWatch Utilities/bin"/cwregister2 --clobber --match=sea_surface_temperature --master=/Users/madisonrichardson/netpp/resources/master_file_sst.nc /Users/madisonrichardson/netpp/data/noaa21/sst_2k/2023/20230502120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc /Users/madisonrichardson/netpp/work/nasa_9k.hdf
Regrid NASA file 0
Add lat/lon 0


[INFO] Creating output /Users/madisonrichardson/netpp/work/nasa_9k.nc
[INFO] Writing sea_surface_temperature
[INFO] Writing latitude
[INFO] Writing longitude


Convert  to NetCDF 0
Run ncgen 0
Compress ofile 0
Archive ofile 0
NetCDF file 'sst_leo_9km_20230502_daily.nc' archived at /Users/madisonrichardson/netpp/data/noaa21/sst/2023/sst_leo_9km_20230502_daily.nc
Processing this date 2023-05-03 00:00:00
overwriting sst_leo_9km_20230503_daily.nc


--2024-10-25 10:18:32--  https://coastwatch.noaa.gov/erddap/files/noaacwLEOACSPOSSTL3SKDaily/2023/123/20230503120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc
Resolving coastwatch.noaa.gov (coastwatch.noaa.gov)... 140.90.107.14, 140.90.107.15
Connecting to coastwatch.noaa.gov (coastwatch.noaa.gov)|140.90.107.14|:443... connected.
HTTP request sent, awaiting response... 200 
Length: 449081018 (428M) [application/x-netcdf]
Saving to: ‘/Users/madisonrichardson/netpp/data/noaa21/sst_2k/2023/20230503120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc.1’

     0K .......... .......... .......... .......... ..........  0%  297K 24m36s
    50K .......... .......... .......... .......... ..........  0%  691K 17m36s
   100K .......... .......... .......... .......... ..........  0% 7.41M 12m3s
   150K .......... .......... .......... .......... ..........  0%  677K 11m44s
   200K .......... .......... .......... .......... ..........  0% 6.02M 9m37s
   

KeyboardInterrupt: 